In [1]:
import pandas as pd
jobs_data = pd.read_csv("job-records_combined.csv.gz", compression='gzip', low_memory=False)
jobs_data.head()

    

,BASE_HASH,CITY,COMPANY_ID,COMPANY_NAME,COUNTRY,CREATED,DELETE_DATE,JOB_HASH,LAST_CHECKED,LAST_UPDATED,STATE,TITLE,UNMAPPED_LOCATION,URL,ZIP
0,104fb04ee3865bddd564238fcd3179b1,collierville,17354,FedEx Corp,USA,2025-03-05 02:26:00+00:00,2025-03-12 05:58:00.000,104fb04ee3865bddd564238fcd3179b1,2025-03-11 05:26:00.000,NaN,TN,Electronics Technician I,False,https://careers.fedex.com/electronics-technici...,38017
1,104fc977eaf83ba557a389f567274c27,germantown,37832,Acadia Healthcare Co Inc,USA,2025-03-11 13:02:00+00:00,2025-03-24 16:18:00.000,104fc977eaf83ba557a389f567274c27,2025-03-24 04:18:00.000,NaN,TN,Mobile Assessment Manager,False,https://non-corporate-acadiahealthcare.icims.c...,38138
2,104fde12c2d8ccf8d5bb7137e9996206,memphis,9609,White Cap Supply Holdings Inc,USA,2025-01-07 17:31:00+00:00,2025-01-13 19:29:00.000,104fde12c2d8ccf8d5bb7137e9996206,2025-01-11 19:16:00.000,NaN,TN,Driver - CDL,False,https://whitecap.wd1.myworkdayjobs.com/careers...,38103
3,104ff44539161718936dfc69b03f273f,Ridgely,79268,Army National Guard of the United States,USA,2026-04-22 05:39:00+00:00,2026-04-23 13:49:00.000,104ff44539161718936dfc69b03f273f,2026-04-22 05:39:00.000,NaN,TN,Licensed Practical Nurse (LPN)- $7500 Sign On ...,False,https://jobs.nationalguard.com/job/759219/lice...,38080
4,104ffd6d2c48f24d7abd463a5d239d2c,nashville,28792,A. O. Smith Corp,USA,2025-11-07 07:26:00+00:00,2026-02-06 01:54:00.000,104ffd6d2c48f24d7abd463a5d239d2c,2026-02-05 01:42:00.000,2025-11-17 08:16:00.000,TN,Senior Automation Project Manager - Manufactur...,False,https://jobs.aosmith.com/job/Nashville-Senior-...,37201


In [2]:
df = pd.read_csv("JobDataCleanWithCountyNames.csv", index_col = 0, header = 0)
df.head()

,CITY,COMPANY_NAME,TITLE,URL,ZIP,COUNTYNAME,DATE_TIME,EXTRACTED_VALUE_HIGH,EXTRACTED_VALUE_LOW,FREQUENCY,NORMALIZED_ANNUAL_HIGH_USD,NORMALIZED_ANNUAL_LOW_USD,NORMALIZED_FREQUENCY
Unnamed: 0,,,,,,,,,,,,,
0,collierville,FedEx Corp,Electronics Technician I,https://careers.fedex.com/electronics-technici...,38017,Fayette County,2025-03-05,32.1,16.92,hourly,66768.0,35193.6,hourly
1,knoxville,Walmart Inc,(USA) Personal Shopper - Sam's,https://careers.walmart.com/us/jobs/WD2118902,37902,Knox County,2025-03-06,24.0,17.00,hourly,49920.0,35360.0,hourly
2,signal mountain,AccentCare Inc,Private Caregiver,https://careers.accentcare.com/jobs/39256?lang...,37377,Hamilton County,2024-03-10,NaN,15.00,hourly,NaN,31200.0,hourly
3,NaN,CVS Health Corp,Lead Director Finance Analytics,https://cvshealth.wd1.myworkdayjobs.com/CVS_He...,37201,Davidson County,2026-07-13,231540.0,100000.00,yearly,231540.0,100000.0,yearly
4,memphis,DHL Group,Order Filler Picker,https://careers.dhl.com/amer/en/job/10281586,38103,Shelby County,2024-10-18,NaN,18.55,hourly,NaN,38584.0,hourly


In [3]:
pip install folium pandas requests

Note: you may need to restart the kernel to use updated packages.


In [4]:
import json
import folium
import pandas as pd
import requests

In [7]:
county_column = "COUNTYNAME"

In [8]:
raw_df = df

In [9]:
raw_df[county_column] = (
    raw_df[county_column]
    .astype(str)
    .str.strip()
    .str.title()
    .str.replace(" County", "", case=False)
)

In [11]:
df1 = raw_df[county_column].value_counts().reset_index()
df1.columns = ["County", "Frequency"]

In [13]:
offline_geojson = {
    "type": "FeatureCollection",
    "features": [
        {
            "type": "Feature",
            "properties": {"name": row["County"]},
            "geometry": {
                "type": "Polygon",
                "coordinates": [
                    [
                        [-90.0, 35.0],
                        [-82.0, 35.0],
                        [-82.0, 36.7],
                        [-90.0, 36.7],
                        [-90.0, 35.0],
                    ]
                ],
            },
        }
        for _, row in df1.iterrows()
    ],
}

In [14]:
m = folium.Map(location=[35.5175, -86.5804], zoom_start=7)

In [15]:
folium.Choropleth(
    geo_data=offline_geojson,
    name="choropleth",
    data=df1,
    columns=["County", "Frequency"],
    key_on="feature.properties.name",
    fill_color="YlOrRd",  
    fill_opacity=0.7,
    line_opacity=0.4,
    legend_name="County Intensity Frequency",
).add_to(m)

In [18]:
m

In [19]:
import math
import folium
import pandas as pd

In [21]:
tn_county_centers = {
    "Anderson": [36.11, -84.20], "Bedford": [35.51, -86.45], "Benton": [36.03, -88.07],
    "Bledsoe": [35.59, -85.20], "Blount": [35.68, -83.92], "Bradley": [35.15, -84.86],
    "Campbell": [36.40, -84.15], "Cannon": [35.81, -86.06], "Carroll": [35.99, -88.36],
    "Carter": [36.30, -82.12], "Cheatham": [36.27, -87.06], "Chester": [35.43, -88.60],
    "Claiborne": [36.48, -83.66], "Clay": [36.55, -85.54], "Cocke": [35.93, -83.11],
    "Coffee": [35.49, -86.07], "Crockett": [35.79, -89.13], "Cumberland": [35.95, -85.00],
    "Davidson": [36.17, -86.78], "Decatur": [35.60, -88.11], "DeKalb": [35.98, -85.83],
    "Dickson": [36.08, -87.38], "Dyer": [36.05, -89.40], "Fayette": [35.20, -89.41],
    "Fentress": [36.38, -84.93], "Franklin": [35.15, -86.10], "Gibson": [35.99, -88.94],
    "Giles": [35.20, -87.03], "Grainger": [36.28, -83.51], "Greene": [36.13, -82.85],
    "Grundy": [35.40, -85.72], "Hamblen": [36.22, -83.27], "Hamilton": [35.18, -85.24],
    "Hancock": [36.53, -83.22], "Hardeman": [35.21, -88.99], "Hardin": [35.20, -88.19],
    "Hawkins": [36.44, -82.95], "Haywood": [35.58, -89.26], "Henderson": [35.65, -88.39],
    "Henry": [36.32, -88.32], "Hickman": [35.80, -87.47], "Houston": [36.28, -87.72],
    "Humphreys": [36.03, -87.76], "Jackson": [36.35, -85.67], "Jefferson": [36.05, -83.45],
    "Johnson": [36.45, -81.86], "Knox": [35.99, -83.94], "Lake": [36.33, -89.49],
    "Lauderdale": [35.76, -89.64], "Lawrence": [35.22, -87.39], "Lewis": [35.52, -87.52],
    "Lincoln": [35.15, -86.58], "Loudon": [35.73, -84.31], "McMinn": [35.42, -84.62],
    "McNairy": [35.17, -88.56], "Macon": [36.53, -86.01], "Madison": [35.62, -88.84],
    "Marion": [35.14, -85.62], "Marshall": [35.47, -86.77], "Maury": [35.62, -87.05],
    "Meigs": [35.51, -84.81], "Monroe": [35.43, -84.25], "Montgomery": [36.50, -87.38],
    "Moore": [35.28, -86.36], "Morgan": [36.14, -84.64], "Obion": [36.36, -89.15],
    "Overton": [36.34, -85.34], "Perry": [35.64, -87.88], "Pickett": [36.56, -85.08],
    "Polk": [35.13, -84.40], "Putnam": [36.14, -85.44], "Rhea": [35.61, -84.86],
    "Roane": [35.86, -84.52], "Robertson": [36.53, -86.87], "Rutherford": [35.84, -86.41],
    "Scott": [36.43, -84.51], "Sequatchie": [35.37, -85.41], "Sevier": [35.78, -83.52],
    "Shelby": [35.18, -89.89], "Smith": [36.25, -85.96], "Stewart": [36.50, -87.84],
    "Sullivan": [36.51, -82.54], "Sumner": [36.47, -86.46], "Tipton": [35.49, -89.76],
    "Trousdale": [36.40, -86.16], "Unicoi": [36.10, -82.43], "Union": [36.28, -83.89],
    "Van Buren": [35.69, -85.46], "Warren": [35.68, -85.78], "Washington": [36.29, -82.49],
    "Wayne": [35.24, -87.79], "Weakley": [36.30, -88.88], "White": [35.93, -85.46],
    "Williamson": [35.92, -86.86], "Wilson": [36.16, -86.30]
}
data_dict = dict(zip(df1["County"], df1["Frequency"]))
max_freq = max(data_dict.values()) if data_dict else 1

In [22]:
import math

def get_color_gradient(value, max_val):
    """Blends Hex values smoothly: Pale Yellow -> Deep Orange -> Intense Crimson Red"""
    if max_val <= 0: return "#FFFFCC"
    ratio = max(0.0, min(1.0, float(value) / float(max_val)))
    
    if ratio < 0.5:
        sub_ratio = ratio / 0.5
        r, g, b = 255, int(255 - (102 * sub_ratio)), int(204 - (204 * sub_ratio))
    else:
        sub_ratio = (ratio - 0.5) / 0.5
        r, g, b = int(255 - (102 * sub_ratio)), int(153 - (153 * sub_ratio)), 0
        
    return f"#{r:02x}{g:02x}{b:02x}"

In [23]:
m = folium.Map(location=[35.8, -86.2], zoom_start=7, tiles="OpenStreetMap")

In [24]:
for name, coords in tn_county_centers.items():
    freq = data_dict.get(name, 0)
    hex_color = get_color_gradient(freq, max_freq)

In [26]:
base_alpha = 0.20
fill_alpha = base_alpha + ((freq / max_freq) * 0.65) if max_freq > 0 else base_alpha

In [27]:
    radius_meters = 13000 + int(math.sqrt(freq) * 1200) if freq > 0 else 8000

    folium.Circle(
        location=coords,
        radius=radius_meters,
        popup=f"<b>County:</b> {name}<br><b>Frequency Count:</b> {freq}",
        tooltip=f"{name}: {freq}",
        color=hex_color,
        weight=2,
        opacity=fill_alpha + 0.1,
        fill=True,
        fill_color=hex_color,
        fill_opacity=fill_alpha,
    ).add_to(m)

In [29]:
m.invalidate_caches() if hasattr(m, 'invalidate_caches') else None

In [30]:
m

In [31]:
base_alpha = 0.50
fill_alpha = base_alpha + ((freq / max_freq) * 0.35) if max_freq > 0 else base_alpha

In [32]:
radius_meters = 13000 + int(math.sqrt(freq) * 1200) if freq > 0 else 8000

In [33]:
    folium.Circle(
        location=coords,
        radius=radius_meters,
        popup=f"<b>County:</b> {name}<br><b>Frequency Count:</b> {freq}",
        tooltip=f"{name}: {freq}",
        color=hex_color,
        weight=2.5,
        opacity=fill_alpha + 0.05,
        fill=True,
        fill_color=hex_color,
        fill_opacity=fill_alpha,
    ).add_to(m)

In [34]:
m

In [37]:
m = folium.Map(location=[35.8, -86.2], zoom_start=8, tiles="OpenStreetMap")

In [38]:
radius_meters = 4000 + int(math.sqrt(freq) * 300) if freq > 0 else 2500

In [39]:
m

In [40]:
m = folium.Map(location=[35.8, -86.2], zoom_start=7, tiles="OpenStreetMap")

for name, coords in tn_county_centers.items():
    freq = data_dict.get(name, 0)
    hex_color = get_color_gradient(freq, max_freq)

In [41]:
base_alpha = 0.55
fill_alpha = base_alpha + ((freq / max_freq) * 0.35) if max_freq > 0 else base_alpha

In [42]:
pixel_radius = 5 + int((freq / max_freq) * 20) if max_freq > 0 else 5

In [43]:
folium.CircleMarker(
        location=coords,
        radius=pixel_radius,
        popup=f"<b>County:</b> {name}<br><b>Frequency Count:</b> {freq}",
        tooltip=f"{name}: {freq}",
        color=hex_color,
        weight=2,
        opacity=fill_alpha + 0.05,
        fill=True,
        fill_color=hex_color,
        fill_opacity=fill_alpha,
    ).add_to(m)

In [44]:
m

In [45]:
print("COUNTY NAMES")
print(df1["County"].unique()[:10])

print("\n--- YOUR TOP COUNTS ---")
print(df1.head())

COUNTY NAMES
['Nan' 'Davidson' 'Shelby' 'Knox' 'Hamilton' 'Rutherford' 'Montgomery'
 'Sullivan' 'Blount' 'Smith']

--- YOUR TOP COUNTS ---
     County  Frequency
0       Nan     156882
1  Davidson      67404
2    Shelby      33263
3      Knox      17528
4  Hamilton      13742


In [46]:
df1 = raw_df[county_column].value_counts().reset_index()
df1.columns = ["County", "Frequency"]

In [47]:
df1 = df1[df1["County"] != "Nan"].reset_index(drop=True)

data_dict = dict(zip(df1["County"], df1["Frequency"]))
max_freq = max(data_dict.values()) if data_dict else 1

In [48]:
m

In [49]:
m = folium.Map(location=[35.8, -86.2], zoom_start=8, tiles="OpenStreetMap")

for name, coords in tn_county_centers.items():
    freq = data_dict.get(name, 0)
    hex_color = get_color_gradient(freq, max_freq)
    
    base_alpha = 0.55
    fill_alpha = base_alpha + ((freq / max_freq) * 0.35) if max_freq > 0 else base_alpha
    
    if freq > 0 and max_freq > 1:
        pixel_radius = 15 + int((math.log(freq) / math.log(max_freq)) * 30)
    else:
        pixel_radius = 12

    folium.CircleMarker(
        location=coords,
        radius=pixel_radius,
        popup=f"<b>County:</b> {name}<br><b>Frequency Count:</b> {freq}",
        tooltip=f"{name}: {freq}",
        color=hex_color,
        weight=2,
        opacity=fill_alpha + 0.05,
        fill=True,
        fill_color=hex_color,
        fill_opacity=fill_alpha,
    ).add_to(m)

In [50]:
m

In [59]:
import branca
import folium
import math

m = folium.Map(location=[35.8, -86.2], zoom_start=8, tiles="OpenStreetMap")

legend_scale = branca.colormap.LinearColormap(
    colors=["#FFFFCC", "#FF9900", "#990000"],
    vmin=0,
    vmax=max_freq,
    caption="County Frequency Distribution Scale",
)
legend_scale.add_to(m)

for name, coords in tn_county_centers.items():
    freq = data_dict.get(name, 0)
    hex_color = get_color_gradient(freq, max_freq)

    base_alpha = 0.55
    fill_alpha = base_alpha + ((freq / max_freq) * 0.35) if max_freq > 0 else base_alpha

    if freq > 0 and max_freq > 1:
        pixel_radius = 15 + int((math.log(freq) / math.log(max_freq)) * 30)
    else:
        pixel_radius = 12

    text_color = "#FF8C00" if (freq > 0 and 0.13 <= (freq / max_freq) < 0.8) else ("#FFD700" if hex_color.upper() == "#FFFFCC" or (freq > 0 and (freq / max_freq) < 0.13) else hex_color)
    
    tooltip_html = f"""
    <div style="font-family: sans-serif; font-size: 13px; line-height: 1.4; padding: 4px;">
        <b style="font-size: 14px; color: #333;">{name} County</b><br>
        <span style="color: #666;">Data Intensity:</span> 
        <b style="color: {text_color if freq > 0 else '#888'};">{freq:,} entries</b>
    </div>
    """

    folium.CircleMarker(
        location=coords,
        radius=pixel_radius,
        popup=f"<b>County:</b> {name}<br><b>Frequency Count:</b> {freq}",
        tooltip=folium.Tooltip(tooltip_html, sticky=True),
        color=hex_color,
        weight=2,
        opacity=fill_alpha + 0.05,
        fill=True,
        fill_color=hex_color,
        fill_opacity=fill_alpha,
    ).add_to(m)


In [60]:
m